# M7.5 — Task-specific lazy datasets

Plan: [`plans/milestone_07/07_catalog_task_dataset_plan.md`](../../plans/milestone_07/07_catalog_task_dataset_plan.md).  
Next: `07_6_catalog_subset_validation.ipynb`.

Catalog rows are task-agnostic. `DatasetTaskSpec` selects participating rows and X/Y fields; `CatalogTaskDataset` exposes **`x, y = dataset[i]`** (dicts). Role images load lazily on access.

**M7 pin:** this notebook sets `image_representation=M7_IMAGE_REPRESENTATION` (`jpeg_uint8`) for historical JPG fixtures. Package default is float `.raw.tif` (`raw_float`) for M8+.

Validation corpus images are grayscale → `(V, 1, H, W)`; contract remains `(V, C, H, W)`.


In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[dl,dev]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ROOT=.


In [2]:
from tomography_ml_validation.milestone_07 import validation_fixture_paths

paths = validation_fixture_paths()
VALIDATION_ROOT = paths["validation_root"]
WORKBOOK_PATH = paths["workbook_path"]
OUTPUT_ROOT = paths["output_root"]
CACHE_ROOT = paths["cache_root"]
print(f"workbook={display_path(WORKBOOK_PATH)}")


workbook=venv/lib/python3.12/site-packages/tomography_ml_validation/test_data/configs/m6/m6_matrix_plan.xlsx


In [3]:
from tomography_ml.gummybear_data_catalog import (
    build_catalog_rows,
    build_task_dataset,
    filter_schedule_consistent,
    load_catalog_jobs,
)
from tomography_ml.gummybear_data_catalog.task_dataset import (
    DatasetTaskSpec,
    M7_IMAGE_REPRESENTATION,
)
import tomography_ml_validation.milestone_07.validation as m7_validation
from tomography_ml_validation.milestone_07 import (
    build_demo_multi_particle_catalog_rows,
    summarize_task_sample,
    write_demo_multi_particle_workbook,
)
from gummybear_validation.notebook_tools import run_installed_pytest_test


## Example A — inpainting (`observed_ref` → `clean_ref`)


In [4]:
catalog_jobs = load_catalog_jobs(WORKBOOK_PATH, VALIDATION_ROOT)
catalog_rows = build_catalog_rows(
    filter_schedule_consistent(catalog_jobs, camera_schedule_id="orbit_matrix_012")
)
inpainting = build_task_dataset(
    catalog_rows,
    DatasetTaskSpec(
        name="inpainting",
        row_filter={"split": "train", "field_status": "complete"},
        x_fields=("observed_ref",),
        y_fields=("clean_ref",),
        image_representation=M7_IMAGE_REPRESENTATION,
    ),
)
summarize_task_sample(inpainting, 0)


{'task': 'inpainting',
 'selected_row_count': 2,
 'index': 0,
 'x': {'observed_ref': {'shape': (12, 1, 128, 128), 'dtype': 'uint8'}},
 'y': {'clean_ref': {'shape': (12, 1, 128, 128), 'dtype': 'uint8'}}}

## Example B — particle localization (single particle)


In [5]:
localization = build_task_dataset(
    catalog_rows,
    DatasetTaskSpec(
        name="localization",
        row_filter={"split": "train", "field_status": "complete"},
        x_fields=("particle_ref",),
        y_fields=("particle_x", "particle_y", "particle_z", "particle_radius"),
        image_representation=M7_IMAGE_REPRESENTATION,
    ),
)
sample = summarize_task_sample(localization, 0)
assert localization.rows[0].n_particles == 1
sample


{'task': 'localization',
 'selected_row_count': 2,
 'index': 0,
 'x': {'particle_ref': {'shape': (12, 1, 128, 128), 'dtype': 'uint8'}},
 'y': {'particle_x': -1.118059158325195,
  'particle_y': 0.4537315368652344,
  'particle_z': 2.5,
  'particle_radius': 3.0}}

## Example B2 — multi-particle labels (`particles`, not scalars)


In [6]:
multi_path = ROOT / "data" / "generated" / "_tmp_m7_multi_particle_task.xlsx"
write_demo_multi_particle_workbook(
    multi_path,
    particle_group_id="task_two_sphere",
    centers=[(-5.0, 0.5, 2.5), (5.0, -0.5, 2.5)],
    repo_root=ROOT,
)
multi_rows = build_demo_multi_particle_catalog_rows(multi_path, repo_root=ROOT)
multi_dataset = build_task_dataset(
    multi_rows,
    DatasetTaskSpec(
        name="multi_particle_localization",
        row_filter={},
        x_fields=("sequence_id", "n_particles"),
        y_fields=("particles", "n_particles"),
    ),
)
_, y = multi_dataset[0]
assert y["n_particles"] == 2 and len(y["particles"]) == 2
print("multi-particle localisation label checks passed")


multi-particle localisation label checks passed


## Example C — shadow forecasting (invert X/Y vs localization)


In [7]:
shadow = build_task_dataset(
    catalog_rows,
    DatasetTaskSpec(
        name="shadow",
        row_filter={"split": "train", "field_status": "complete"},
        x_fields=("particle_x", "particle_y", "particle_z", "particle_radius"),
        y_fields=("particle_ref",),
        image_representation=M7_IMAGE_REPRESENTATION,
    ),
)
summarize_task_sample(shadow, 0)


{'task': 'shadow',
 'selected_row_count': 2,
 'index': 0,
 'x': {'particle_x': -1.118059158325195,
  'particle_y': 0.4537315368652344,
  'particle_z': 2.5,
  'particle_radius': 3.0},
 'y': {'particle_ref': {'shape': (12, 1, 128, 128), 'dtype': 'uint8'}}}

## Task dataset validation


In [8]:
run_installed_pytest_test(
    m7_validation,
    "test_m7_5_task_dataset_returns_lazy_xy_without_redefining_sample_as_tensor",
)


M7.5
Test executed: test_m7_5_task_dataset_returns_lazy_xy_without_redefining_sample_as_tensor()

pytest:
../../venv/lib/python3.12/site-packages/tomography_ml_validation/milestone_07/validation.py . [100%]
============================== 1 passed in 1.73s ===============================

Test proves: Catalog rows convert to task-specific lazy (x, y) by selecting rows and fields —
             not by redefining the sample as a tensor.
